**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

**Check motifdelta**

In [2]:
${FP_APP} python - <<'PY'
import motifdelta
import pkgutil
print("motifdelta path:", motifdelta.__path__)
print("submodules:", [m.name for m in pkgutil.iter_modules(motifdelta.__path__)])
PY

motifdelta path: ['/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/motifdelta/src/motifdelta']
submodules: ['score', 'seq', 'stats']


## Prepare

In [3]:
### choose between biostat and igvf
CHOOSE_PARTITION="biostat"
#CHOOSE_PARTITION="igvf"

if [[ "$CHOOSE_PARTITION" == "igvf" ]]; then
    SLURM_ACCOUNT="majoroslab"
    SLURM_PARTITION="igvf,common"
elif [[ "$CHOOSE_PARTITION" == "biostat" ]]; then
    SLURM_ACCOUNT="biostat"
    SLURM_PARTITION="biostat"
else
    echo "Unknown CHOOSE_PARTITION: $CHOOSE_PARTITION"
    exit 1
fi

echo ${SLURM_ACCOUNT}
echo ${SLURM_PARTITION}

biostat
biostat


## JASPAR2024
`JASPAR2024_CORE_vertebrates_non-redundant.meme.pkl`

### Prepare

In [4]:
ls -1 ${FD_DATA}/motif_jaspar2024

JASPAR2024_CORE_non-redundant_pfms_jaspar.zip
JASPAR2024_CORE_non-redundant_pfms_meme.zip
JASPAR2024_CORE_non-redundant_pfms_transfac.zip
JASPAR2024_CORE_vertebrates_meme
JASPAR2024_CORE_vertebrates_non-redundant.meme
JASPAR2024_CORE_vertebrates_non-redundant.meme.pkl
JASPAR2024_CORE_vertebrates_non-redundant.npz
JASPAR2024_CORE_vertebrates_non-redundant_pfms_jaspar.zip
JASPAR2024_CORE_vertebrates_non-redundant_pfms_meme.zip
JASPAR2024_CORE_vertebrates_non-redundant_pfms_transfac.zip


In [5]:
TXT_FDIRY_INP="/hpc/group/igvf/kk319/data/motif_jaspar2024"
TXT_FNAME_INP="JASPAR2024_CORE_vertebrates_non-redundant.meme.pkl"
TXT_FPATH_INP=${TXT_FDIRY_INP}/${TXT_FNAME_INP}

TXT_FDIRY_OUT="/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard"
TXT_FNAME_BKG="background_zero_order.npy"
TXT_FPATH_BKG=${TXT_FDIRY_OUT}/${TXT_FNAME_BKG}

TXT_FNAME_OUT="motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl"
TXT_FPATH_OUT=${TXT_FDIRY_OUT}/${TXT_FNAME_OUT}

ls -lh ${TXT_FPATH_INP}
ls -lh ${TXT_FPATH_BKG}

-rw-r--r--. 1 kk319 majoroslab 316K Feb 23 15:38 /hpc/group/igvf/kk319/data/motif_jaspar2024/JASPAR2024_CORE_vertebrates_non-redundant.meme.pkl
-rw-r--r--. 1 kk319 majoroslab 160 Feb 23 15:33 /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/background_zero_order.npy


### Execute

In [6]:
### set script
FP_EXE=${FD_EXE}/run_motif_meme2lods.py

### set log file
FP_LOG=${FD_LOG}/run_motif_meme2lods_jaspar2024.txt

### set resource
NUM_CPU=2
NUM_MEM=4G

SLURM_OPTS=(
  -A "${SLURM_ACCOUNT}"
  -p "${SLURM_PARTITION}"
  --job-name=meme2lods_jaspar2024
  --cpus-per-task="${NUM_CPU}"
  --mem="${NUM_MEM}"
  --output="${FP_LOG}"
  --chdir="${FD_EXE}"
  --export=ALL,FP_CNF="${FP_CNF}"
  --parsable
)

### execute
SLURM_JOBID=$(sbatch "${SLURM_OPTS[@]}" <<EOF
#!/bin/bash
set -euo pipefail

### init
timer_start=\$(date +%s)
source "${FP_CNF}"

### print start message
echo "Hostname:   \$(hostname)"
echo "Time Stamp: \$(date +"%m-%d-%y+%T")" 
echo "PWD:    \$(pwd)"
echo "FP_APP: ${FP_APP}"
echo "FD_EXE: ${FD_EXE}"
echo "PYTHONPATH (host): \${PYTHONPATH:-<empty>}"
echo

### execute
echo "=== Run main script ==="
${FP_APP} python ${FP_EXE} \
  --txt_fpath_inp ${TXT_FPATH_INP} \
  --txt_fpath_bkg ${TXT_FPATH_BKG} \
  --txt_fpath_out ${TXT_FPATH_OUT}

### print end message
timer=\$(date +%s)
runtime=\$(( timer - timer_start ))
echo
echo 'Done!'
echo "Run Time: \$(displaytime \${runtime})"
EOF
)

echo "Submitted job: ${SLURM_JOBID}"

Submitted job: 43673705


### Review

In [7]:
#sacct_summary.sh 41759295
sacct_summary.sh ${SLURM_JOBID}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
43673705.ba+                          batch  COMPLETED   00:00:12  00:01.096     82744K                 dcc-biostat-20 

===== ElapsedRaw =====
ElapsedRaw = 12 sec (0.20 min)

===== MaxRSS =====
MaxRSS = 0.08 GiB


In [8]:
cat ${FD_LOG}/run_motif_meme2lods_jaspar2024.txt

Hostname:   dcc-biostat-20
Time Stamp: 02-23-26+18:12:36
PWD:    /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
FP_APP: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/run_script.sh
FD_EXE: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PYTHONPATH (host): <empty>

=== Run main script ===
Loaded background: [0.31068275 0.18781467 0.18880767 0.31269491]
Loaded 879 motifs
Example keys: ['MA0002.3 Runx1', 'MA0003.5 TFAP2A', 'MA0004.1 Arnt']
First PWM shape: (9, 4)
Saved 879 motifs -> /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl

Done!
Run Time: 1 seconds


In [9]:
${FP_APP} python - <<'PY'
import pickle, numpy as np

fp = "/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl"
with open(fp, "rb") as f:
    d = pickle.load(f)

print("keys:", d.keys())
print("alphabet:", d.get("alphabet"))
print("bg:", d["bg"], "sum=", np.sum(d["bg"]))
print("num names:", len(d["names"]))
print("num pwms:", len(d["pwms"]), "num lods:", len(d["lods"]))

k = d["names"][0]
print("example motif:", k)
print("pwm shape:", d["pwms"][k].shape, "lod shape:", d["lods"][k].shape)
print("pwm row sums (first 3):", d["pwms"][k][:3].sum(axis=1))
PY


keys: dict_keys(['pwms', 'lods', 'bg', 'names', 'alphabet'])
alphabet: ACGT
bg: [0.31068275 0.18781467 0.18880767 0.31269491] sum= 1.0
num names: 879
num pwms: 879 num lods: 879
example motif: MA0002.3 Runx1
pwm shape: (9, 4) lod shape: (9, 4)
pwm row sums (first 3): [1. 1. 1.]


## Non-redundant motifs
`Non-redundant motifs (jvierstra; v2.1 beta)`

### Prepare

In [10]:
ls -1 ${FD_DATA}/motif_nonredundant_jvierstra_v2.1beta

all.dbs.meme.pkl
all.dbs.meme.txt
consensus_pwms.meme.pkl
consensus_pwms.meme.txt
metadata.tsv
tomtom.all.txt


In [11]:
TXT_FDIRY_INP="${FD_DATA}/motif_nonredundant_jvierstra_v2.1beta"
TXT_FNAME_INP="consensus_pwms.meme.pkl"
TXT_FPATH_INP=${TXT_FDIRY_INP}/${TXT_FNAME_INP}

TXT_FDIRY_OUT="${FD_RES}/analysis_variant_motif_richard"
TXT_FNAME_BKG="background_zero_order.npy"
TXT_FPATH_BKG=${TXT_FDIRY_OUT}/${TXT_FNAME_BKG}

TXT_FNAME_OUT="motif_nonredundant_jvierstra_v2.1beta.lods.pkl"
TXT_FPATH_OUT=${TXT_FDIRY_OUT}/${TXT_FNAME_OUT}

ls -lh ${TXT_FPATH_INP}
ls -lh ${TXT_FPATH_BKG}

-rw-r--r--. 1 kk319 majoroslab 292K Feb 23 15:38 /hpc/group/igvf/kk319/data/motif_nonredundant_jvierstra_v2.1beta/consensus_pwms.meme.pkl
-rw-r--r--. 1 kk319 majoroslab 160 Feb 23 15:33 /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/background_zero_order.npy


### Execute

In [12]:
### set script
FP_EXE=${FD_EXE}/run_motif_meme2lods.py

### set log file
FP_LOG=${FD_LOG}/run_motif_meme2lods_jvierstra.txt

### set resource
NUM_CPU=2
NUM_MEM=4G

SLURM_OPTS=(
  -A "${SLURM_ACCOUNT}"
  -p "${SLURM_PARTITION}"
  --job-name=run_meme2lods_jvierstra
  --cpus-per-task="${NUM_CPU}"
  --mem="${NUM_MEM}"
  --output="${FP_LOG}"
  --chdir="${FD_EXE}"
  --export=ALL,FP_CNF="${FP_CNF}"
  --parsable
)

### execute
SLURM_JOBID=$(sbatch "${SLURM_OPTS[@]}" <<EOF
#!/bin/bash
set -euo pipefail

### init
timer_start=\$(date +%s)
source "${FP_CNF}"

### print start message
echo "Hostname:   \$(hostname)"
echo "Time Stamp: \$(date +"%m-%d-%y+%T")" 
echo "PWD:    \$(pwd)"
echo "FP_APP: ${FP_APP}"
echo "FD_EXE: ${FD_EXE}"
echo "PYTHONPATH (host): \${PYTHONPATH:-<empty>}"
echo

### execute
echo "=== Run main script ==="
${FP_APP} python ${FP_EXE} \
  --txt_fpath_inp ${TXT_FPATH_INP} \
  --txt_fpath_bkg ${TXT_FPATH_BKG} \
  --txt_fpath_out ${TXT_FPATH_OUT}

### print end message
timer=\$(date +%s)
runtime=\$(( timer - timer_start ))
echo
echo 'Done!'
echo "Run Time: \$(displaytime \${runtime})"
EOF
)

echo "Submitted job: ${SLURM_JOBID}"

Submitted job: 43673734


### Review

In [13]:
#sacct_summary.sh 41760496
sacct_summary.sh ${SLURM_JOBID}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
43673734.ba+                          batch  COMPLETED   00:00:10  00:01.067     84052K                 dcc-biostat-20 

===== ElapsedRaw =====
ElapsedRaw = 10 sec (0.17 min)

===== MaxRSS =====
MaxRSS = 0.08 GiB


In [14]:
cat ${FD_LOG}/run_motif_meme2lods_jvierstra.txt

Hostname:   dcc-biostat-20
Time Stamp: 02-23-26+18:13:36
PWD:    /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
FP_APP: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/run_script.sh
FD_EXE: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PYTHONPATH (host): <empty>

=== Run main script ===
Loaded background: [0.31068275 0.18781467 0.18880767 0.31269491]
Loaded 637 motifs
Example keys: ['AC0001:GATA/PROP:GATA AC0001:GATA/PROP:GATA', 'AC0002:PROP/ALX:Homeodomain AC0002:PROP/ALX:Homeodomain', 'AC0003:HNF1A/HNF1B:Homeodomain AC0003:HNF1A/HNF1B:Homeodomain']
First PWM shape: (13, 4)
Saved 637 motifs -> /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl

Done!
Run Time: 1 seconds


In [15]:
${FP_APP} python - <<'PY'
import pickle, numpy as np

fp = "/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl"
with open(fp, "rb") as f:
    d = pickle.load(f)

print("keys:", d.keys())
print("alphabet:", d.get("alphabet"))
print("bg:", d["bg"], "sum=", np.sum(d["bg"]))
print("num names:", len(d["names"]))
print("num pwms:", len(d["pwms"]), "num lods:", len(d["lods"]))

k = d["names"][0]
print("example motif:", k)
print("pwm shape:", d["pwms"][k].shape, "lod shape:", d["lods"][k].shape)
print("pwm row sums (first 3):", d["pwms"][k][:3].sum(axis=1))
PY

keys: dict_keys(['pwms', 'lods', 'bg', 'names', 'alphabet'])
alphabet: ACGT
bg: [0.31068275 0.18781467 0.18880767 0.31269491] sum= 1.0
num names: 637
num pwms: 637 num lods: 637
example motif: AC0001:GATA/PROP:GATA AC0001:GATA/PROP:GATA
pwm shape: (13, 4) lod shape: (13, 4)
pwm row sums (first 3): [0.999999 1.       1.      ]
